# **Cuaderno 02: Limpieza y Preprocesado Base**

En este cuaderno nos encargaremos de preparar el conjunto de datos `train.csv`. El objetivo principal es transformar las variables en crudo a formatos que faciliten la posterior generación de variables temporales y evitar cualquier tipo de filtración de información.

Las decisiones de diseño principales son:
1. **Ordenación cronológica:** Es imperativo ordenar el dataset por fecha para poder simular el avance real del tiempo.
2. **Codificación del Target:** Transformaremos los resultados a valores numéricos para los algoritmos de Machine Learning.
3. **Generación de columnas base:** Calcularemos los puntos obtenidos por cada equipo en el partido para facilitar el cálculo de rachas futuras.

In [1]:
import pandas as pd
import numpy as np
import os

ruta_datos = '../data/interim/train_enriquecido.csv'
df = pd.read_csv(ruta_datos)

print(f"Dimensiones iniciales: {df.shape}")
df.head()

Dimensiones iniciales: (17480, 19)


,id,division,anio_fin_temporada,fecha,local,visitante,goles_local,goles_visitante,resultado,valor_plantilla_local,valor_plantilla_visitante,altura_media_local,edad_media_local,altura_media_visitante,edad_media_visitante,num_lesionados_local,num_lesionados_visitante,coste_lesionados_local,coste_lesionados_visitante
0,10341_2223_ESP1,ESP_LaLiga,2023,2023-06-04,Espanol,Almeria,3,3,X_empate,125050000.0,74950000.0,179.7,26.83,181.4,25.23,0,1,0.0,1800000.0
1,10334_2223_ESP1,ESP_LaLiga,2023,2023-06-04,Osasuna,Girona,2,1,1_local,115700000.0,39550000.0,181.1,27.66,181.6,26.14,2,4,7000000.0,12100000.0
2,10335_2223_ESP1,ESP_LaLiga,2023,2023-06-04,Real Madrid,Ath Bilbao,1,1,X_empate,866000000.0,216400000.0,181.3,27.21,181.9,26.71,0,4,0.0,38000000.0
3,10336_2223_ESP1,ESP_LaLiga,2023,2023-06-04,Sociedad,Sevilla,2,1,1_local,368600000.0,264700000.0,181.5,26.02,180.7,28.52,1,0,18000000.0,0.0
4,10337_2223_ESP1,ESP_LaLiga,2023,2023-06-04,Villarreal,Ath Madrid,2,2,X_empate,313200000.0,676500000.0,181.1,28.10,181.6,27.44,2,5,30000000.0,151000000.0


## **1. Tratamiento Temporal (Prevención de Data Leakage)**
Convertimos la columna `fecha` a un objeto `datetime` de Pandas. Tras esto, ordenamos los partidos de más antiguo a más reciente y reseteamos el índice. Esto asegura que al aplicar ventanas temporales más adelante, nunca utilicemos datos del futuro.

In [ ]:
df['fecha'] = pd.to_datetime(df['fecha'])

df = df.sort_values(by='fecha')

df = df.reset_index(drop=True)

df[['id', 'fecha', 'local', 'visitante']]

,id,fecha,local,visitante
0,2832_0001_ENG1,2000-08-19,Sunderland,Arsenal
1,2828_0001_ENG1,2000-08-19,Derby,Southampton
2,2825_0001_ENG1,2000-08-19,Charlton,Man City
3,2826_0001_ENG1,2000-08-19,Chelsea,West Ham
4,2831_0001_ENG1,2000-08-19,Liverpool,Bradford
...,...,...,...,...
17475,10339_2223_ESP1,2023-06-04,Celta,Barcelona
17476,10340_2223_ESP1,2023-06-04,Elche,Cadiz
17477,10333_2223_ESP1,2023-06-04,Mallorca,Vallecano
17478,10338_2223_ESP1,2023-06-04,Betis,Valencia


## **2. Codificación de la Variable Objetivo (`resultado`)**
El algoritmo de predicción necesitará que nuestra variable objetivo sea numérica. Procedemos a realizar el siguiente mapeo:
* `1_local` -> 0
* `X_empate` -> 1
* `2_visitante` -> 2

*Nota:* Hemos decidido mantener la columna original de momento para facilitar las comprobaciones, creando una nueva llamada `resultado_cod`.

In [3]:
mapa_resultado = {
    '1_local': 0,
    'X_empate': 1,
    '2_visitante': 2
}

df['resultado_cod'] = df['resultado'].map(mapa_resultado)


print("Valores nulos en resultado_cod:", df['resultado_cod'].isnull().sum())

Valores nulos en resultado_cod: 0


## **3. Creación de variables auxiliares: Sistema de Puntuación**
Para facilitar la fase de en la que calcularemos los puntos acumulados por los equipos, vamos a traducir el resultado a los puntos reales obtenidos en ese encuentro por el equipo local y el equipo visitante.

* Victoria: 3 puntos
* Empate: 1 punto
* Derrota: 0 puntos

In [ ]:
mapa_puntos_local = {
    '1_local': 3,
    'X_empate': 1,
    '2_visitante': 0
}

mapa_puntos_visitante = {
    '1_local': 0,
    'X_empate': 1,
    '2_visitante': 3
}

df['puntos_local'] = df['resultado'].map(mapa_puntos_local)
df['puntos_visitante'] = df['resultado'].map(mapa_puntos_visitante)

df[['resultado', 'puntos_local', 'puntos_visitante']].head(5)

,resultado,puntos_local,puntos_visitante
0,1_local,3,0
1,X_empate,1,1
2,1_local,3,0
3,1_local,3,0
4,1_local,3,0


## **4. Exportación a la zona Interim**
Con las fechas ordenadas, la variable objetivo codificada y el sistema de puntuación establecido, hemos finalizado la preparación de los datos. 

Es importante destacar que no eliminamos aún los nombres de los equipos (`local`, `visitante`) ni las fechas, ya que son la clave para agrupar los datos en el próximo cuaderno.


In [ ]:
os.makedirs('../data/interim', exist_ok=True)

ruta_salida = '../data/interim/train_limpio.csv'
df.to_csv(ruta_salida, index=False)

print(f"Dataset guardado en: {ruta_salida}")

Dataset limpio guardado exitosamente en: ../data/interim/train_limpio.csv
